# **최단 경로(Shortest Path)**

<aside>
💡

간선의 가중치가 있는 그래프에서 두 정점 사이의 경로들 중에 간선의 가중치의 합이 최소인 경로

</aside>

그래프에서 두 정점(또는 모든 정점 쌍) 사이를 이동할 때, 가장 적은 비용(거리, 시간 등 가중치)으로 이동하는 경로를 찾는 문제를 **최단 경로**(Shortest Path) 문제라고 한다.

지도를 떠올리면 `“최단 거리로 목적지에 가고 싶다”`와 같은 상황이 직관적인 예시가 될 수 있다.

**다양한 최단 경로 알고리즘**을 살펴보며, **Dijkstra**, **Bellman-Ford**, 그리고 **Floyd-Warshall** 알고리즘을 학습한다.

## **1. 기본 용어 정리**

1. **가중치(Weight)**: 각 간선(edge)이 갖는 비용(거리, 시간, 비용 등)을 의미.
2. **단일 출발 최단 경로(Single-Source Shortest Path)**: 특정 시작점에서 모든 다른 정점으로 가는 최단 경로.
    - Dijkstra 알고리즘
    - Bellman-Ford 알고리즘
3. **전체 쌍 최단 경로(All-Pairs Shortest Path)**: 그래프 내 모든 정점 쌍에 대한 최단 경로.
    - Floyd-Warshall 알고리즘

## **2. Dijkstra 알고리즘**

<aside>
💡

시작 정점에서 다른 모든 정점으로의 최단 경로를 구하는 알고리즘

</aside>

<aside>
📌

음의 가중치가 없는 그래프에서 단일 출발점으로부터 다른 모든 정점으로의 최단 경로를 우선순위 큐를 사용해 효율적으로 찾는 알고리즘

</aside>

### **2.1 개념**

- **최단 경로 알고리즘** 중 가장 널리 사용.
- **음의 가중치**가 없는(가중치 ≥ 0) 그래프에서 **단일 출발 최단 경로**를 빠르게 구한다.
- 원리
    - **한 걸음씩**, 이미 확정된 정점에 인접한 간선 중 **가장 짧은 거리**를 갖는 정점을 골라 최단 거리를 확정해 나감.

### 2.2 Dijkstra 코드

In [ ]:
import heapq

def dijkstra(num_vertices, adj_list, start):
    """
    Dijkstra 알고리즘:
    - 음의 간선이 없는 그래프에서
    - 시작점 'start'로부터 각 정점까지의 최단 거리 반환
    :param num_vertices: 정점 개수 (1~num_vertices)
    :param adj_list: {node: [(cost, adjacent_node), ...]} 형태
    :param start: 시작 정점
    :return: distance 배열 (인덱스: 정점, 값: 최단 거리)
    """
    INF = float('inf')
    distance = [INF] * (num_vertices + 1)
    distance[start] = 0

    # 우선순위 큐 (최소 힙)
    # (현재까지 알려진 최단 거리, 정점)
    priority_queue = [(0, start)]

    while priority_queue:
        curr_dist, node = heapq.heappop(priority_queue)

        # 이미 더 짧은 경로가 확정된 상태라면 skip
        if distance[node] < curr_dist:
            continue

        # node와 인접한 노드들 확인 (간선 완화)
        for edge_cost, adj_node in adj_list[node]:
            new_dist = curr_dist + edge_cost
            # 더 짧은 경로 발견 시, 갱신
            if new_dist < distance[adj_node]:
                distance[adj_node] = new_dist
                heapq.heappush(priority_queue, (new_dist, adj_node))

    return distance


# ---- 사용 예시 ----
n = 5
# 인접 리스트 예 (가중치, 목적지)
# 예: 1번 노드 -> 2번 노드 비용 2, ->4번 노드 비용 5
adjacency_list = {
    1: [(2, 2), (5, 4)],
    2: [(2, 1), (3, 3), (2, 5)],
    3: [(3, 2), (6, 4)],
    4: [(5, 1), (6, 3), (1, 5)],
    5: [(2, 2), (1, 4)],
}

start_node = 1
result_dist = dijkstra(n, adjacency_list, start_node)

for v in range(1, n + 1):
    print(f"{start_node} -> {v} 최단 거리:", result_dist[v])

In [ ]:
# 출력 결과
# 1 -> 1 최단 거리: 0
# 1 -> 2 최단 거리: 2
# 1 -> 3 최단 거리: 5
# 1 -> 4 최단 거리: 5
# 1 -> 5 최단 거리: 4

**해설**

1. **distance** 배열: 각 노드까지의 **현재까지 파악된 최단 거리**
2. **우선순위 큐**: (거리, 노드) 형태로 관리, **가장 작은 거리**를 가진 노드부터 탐색
3. **간선 완화(Relaxation)**: `distance[adj_node]`를 새로운 경로가 더 짧으면 갱신

## **3. Bellman-Ford 알고리즘**

<aside>
💡

시작 정점에서 다른 모든 정점으로의 최단 경로를 구하는 알고리즘

</aside>

<aside>
📌

음수 가중치가 있어도 사용 가능하며, 시작점에서 각 정점까지의 최단 거리를 모든 간선을 반복적으로 검사하여 갱신하는 알고리즘

</aside>

### **3.1 개념**

- **음의 가중치**가 존재할 수 있는 그래프에서, 단일 출발 최단 경로를 구하는 알고리즘
- 단, **음의 사이클(negative cycle)**이 있으면 최단 경로가 정의되지 않음
- Dijkstra알고리즘과 달리, 탐욕 기법 대신 **동적 프로그래밍(DP)** 접근을 사용

### **3.2 동작 과정 (요약)**
    
1. 시작 정점(src)에서의 거리를 0, 나머지는 $∞$(아주 큰 수)로 초기화.
    1. 시작 정점에서 각 정점까지의 최단 거리를 저장할 리스트를 생성
2. 모든 간선 정보를 반복적으로 확인하며, 
(u→v) 간선에 대해 **distance[v] > distance[u] + cost**이면 업데이트(완화)
3. 최대 (정점 수 - 1)번 반복
    1. 마지막 정점을 제외한 모든 정점에 대해서 2번 과정 반복
4. 만약 (정점 수 - 1)번 반복 후에도 업데이트가 발생한다면(거리가 갱신되면), **음의 사이클**이 존재한다고 판단

### Bellman-Ford 코드

In [ ]:
def bellman_ford(num_vertices, edges, start):
    """
    Bellman-Ford 알고리즘:
    - 음의 가중치가 있어도 OK (음의 사이클만 없다면)
    - 단일 출발 최단 경로
    :param num_vertices: 정점 개수
    :param edges: (start_node, end_node, cost) 형식의 간선 리스트
    :param start: 시작 정점
    :return: distance 배열, 음의 사이클 존재시 None 반환
    """
    INF = float('inf')
    distance = [INF] * (num_vertices + 1)
    distance[start] = 0

    # (정점수 -1)번 반복
    for _ in range(num_vertices - 1):
        updated = False
        for u, v, w in edges:
            if distance[u] != INF and distance[v] > distance[u] + w:
                distance[v] = distance[u] + w
                updated = True
        if not updated:  # 더 이상 업데이트 없으면 break
            break

    # 음의 사이클 체크: 한 번 더 확인
    for u, v, w in edges:
        if distance[u] != INF and distance[v] > distance[u] + w:
            # 음의 사이클 존재
            return None

    return distance


# ---- 사용 예시 ----
n = 5
# edges: (u, v, cost)
edges_info = [
    (1, 2, 2),
    (1, 4, 5),
    (2, 3, 3),
    (2, 5, 2),
    (3, 4, 6),
    (4, 5, 1),
    # 예) 음의 간선도 가능
    # (5, 3, -4) -> 음의 사이클 등 테스트
]
start_node = 1
result = bellman_ford(n, edges_info, start_node)

if result is None:
    print("음의 사이클(negative cycle)이 존재합니다.")
else:
    for v in range(1, n + 1):
        print(f"{start_node} -> {v} 최단 거리:", result[v])

In [ ]:
# 출력 결과
# 1 -> 1 최단 거리: 0
# 1 -> 2 최단 거리: 2
# 1 -> 3 최단 거리: 5
# 1 -> 4 최단 거리: 5
# 1 -> 5 최단 거리: 4

**해설**

1. **distance** 배열 초기화(시작 노드=0, 나머지=$∞$)
2. (정점 수-1)번 반복하며 모든 간선에 대해 **완화** 연산
3. 마지막에 한 번 더 검사하여 추가 업데이트가 일어나면 **음의 사이클** 존재 판정
4. Dijkstra와 달리 **음의 가중치**도 처리 가능

## **4. Floyd-Warshall 알고리즘**

<aside>
💡

**모든 정점 쌍** 사이의 **최단 거리**를 구하는 알고리즘

</aside>

### **4.1 개념**

- **모든 정점 쌍** 사이의 **최단 거리**를 구하는 알고리즘 (All-Pairs Shortest Path)
- 음의 가중치도 가능 (단, 음의 사이클이 있으면 최단 경로 정의 불가)
- 동적 프로그래밍을 사용하여 최단 경로를 점진적으로 갱신

### **4.2 동작 원리**
    
- 3중 for문으로, **중간에 거쳐 갈 수 있는 정점** k를 1개씩 늘려가며 최단 거리 갱신
- `distance[i][j] = min(distance[i][j], distance[i][k] + distance[k][j])`
- i, j, k는 각각 모든 정점을 순회

### 4.3 Floyd-Warshall 코드

In [ ]:
def floyd_warshall(num_vertices, graph_matrix):
    """
    Floyd-Warshall 알고리즘:
    - 모든 정점 쌍 최단 거리
    - graph_matrix[i][j] = i->j 간선 가중치 (없으면 INF)
    :param num_vertices: 정점 개수
    :param graph_matrix: 2차원 리스트 (가중치)
    :return: distance (최종 최단 거리 2차원 리스트)
    """
    INF = float('inf')
    # distance를 graph_matrix로 초기화 (깊은 복사)
    distance = [row[:] for row in graph_matrix]

    for k in range(1, num_vertices + 1):
        for i in range(1, num_vertices + 1):
            for j in range(1, num_vertices + 1):
                # 거쳐가는 경로가 더 짧으면 업데이트
                if distance[i][k] + distance[k][j] < distance[i][j]:
                    distance[i][j] = distance[i][k] + distance[k][j]
    return distance


# ---- 사용 예시 ----
n = 4
INF = float('inf')

# graph_matrix[i][j]: i->j 가중치, 없으면 INF
# 정점 인덱스 1~n, 편의상 크기 (n+1)x(n+1)
graph_matrix = [[INF] * (n + 1) for _ in range(n + 1)]
"""
[[inf, inf, inf, inf, inf],
 [inf, inf, inf, inf, inf],
 [inf, inf, inf, inf, inf],
 [inf, inf, inf, inf, inf],
 [inf, inf, inf, inf, inf]]
"""

# 대각선(자기 자신)은 0
for i in range(1, n + 1):
    graph_matrix[i][i] = 0

# 예: 간선 (1->2=4, 1->3=2, 2->3=3, 2->4=2, 3->2=1, 3->4=4, ...)
graph_matrix[1][2] = 4
graph_matrix[1][3] = 2
graph_matrix[2][3] = 3
graph_matrix[2][4] = 2
graph_matrix[3][2] = 1
graph_matrix[3][4] = 4
graph_matrix[4][1] = 1
# 필요시 양방향이면 대칭적으로 값을 설정

result_dist = floyd_warshall(n, graph_matrix)
for i in range(1, n + 1):
    for j in range(1, n + 1):
        if result_dist[i][j] == INF:
            print("INF", end=' ')
        else:
            print(result_dist[i][j], end=' ')
    print()

In [ ]:
# 출력 결과
0 3 2 5 
3 0 3 2 
4 1 0 3 
1 4 3 0 

**해설**

1. `distance[i][j]`를 초기 설정(간선 비용)
2. 모든 k에 대해, i → k → j 경로가 더 짧으면 업데이트
3. 3중 for문 → 시간 복잡도: **`$O(N^3)$`**

## **5. 알고리즘 비교**

| 알고리즘 | **문제 유형** | **시간 복잡도** | **음의 간선** | **특징** |
| --- | --- | --- | --- | --- |
| **Dijkstra** | 단일 출발 최단 경로 (가중치≥0) | `$O(E log V)$` 정도 (힙) | 불가 (가중치≥0) | 가장 널리 사용됨.  |
| **Bellman-Ford** | 단일 출발 최단 경로 (음수 가능) | `$O(VE)$` | 가능 | 음의 사이클도 판별 가능 |
| **Floyd-Warshall** | 전체 쌍 최단 경로 (All-Pairs) | `$O(V^3)$` | 가능 | 모든 정점 쌍. V가 크면 느림. 음의 사이클도 판별 가능 |
- Dijkstra
    - **양의 가중치** 그래프, 단일 출발 → **가장 빠름**(우선순위 큐)
- Bellman-Ford
    - **음수 가중치**(단, 음의 사이클 없음) → 단일 출발
- Floyd-Warshall
    - **모든 정점 쌍** 최단 경로 → `$O(N^3)$`, 음수 가중치도 가능(단, 음의 사이클 없음)
- **Bellman-Ford**와 **Floyd-Warshall** 모두 “음의 사이클(negative cycle)이 존재하는지”를 **판별**할 수 있음
    - 단, **음의 사이클이 실제로 존재**한다면, “**정의된 최단 경로**”를 구할 수는 없다.
    (왜냐하면 음의 사이클을 통해 무한히 비용을 줄일 수 있기 때문).

## **6. 결론**

<aside>
💡

“Dijkstra(양수 간선), Bellman-Ford(음수 간선), Floyd-Warshall(모든 쌍)” 

</aside>

1. **최단 경로**는 “그래프 상 특정 시작 정점에서 다른 정점으로 가는 경로의 비용이 최소가 되는” 문제 혹은 “모든 정점 쌍 간 최소 경로” 문제.
2. 주요 알고리즘
    - **Dijkstra**: 가중치가 **비음수**, 단일 출발
    - **Bellman-Ford**: **음수 가중치** 허용(음의 사이클 판별 가능), 단일 출발
    - **Floyd-Warshall**: 모든 정점 쌍, `$O(N^3)$`
3. 각각의 **자료 구조나 구현 기법**(우선순위 큐, 인접 리스트, 간선 리스트 등)을 적절히 사용해 효율을 높일 수 있음.

**핵심 포인트**

- **문제 상황**(음수 간선, 단일 출발 vs. 모든 쌍, 그래프 크기)에 따라 **알고리즘 선택**.